# Notebook 5: Defense Mechanisms

Demonstrate all 8 defense types against attacks.

**Defenses:** Input Sanitization, Temporal Consistency, Multi-Sensor Agreement, Robust Clustering, Anomaly Detection, Certified Defense, Adversarial Training, Ensemble

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import load_scenario, get_all_detections, get_ground_truth, get_ownship

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 8)
from attacks.camera_attacks import CameraAdversarialAttacker, AttackType
from defenses.defense_mechanisms import DefensePipeline, DefenseType


## 5.1 Load Data and Create Attack

In [ ]:
SCENARIO = 'scenario2'
loader = load_scenario(SCENARIO)
detections = get_all_detections(loader)
ground_truth = get_ground_truth(loader)

attacker = CameraAdversarialAttacker(epsilon=0.05)
attacked = {sid: df.copy() for sid, df in detections.items()}
attacked[3] = attacker.attack_detections(detections[3].copy(), AttackType.FGSM, sensor_id=3)

print('Benign vs Attacked IR Camera:')
print('  Benign: ', len(detections[3]), 'detections')
print('  Attacked:', len(attacked[3]), 'detections')

## 5.2 Initialize Defense Pipeline

In [ ]:
pipeline = DefensePipeline()
print('Available defenses:')
for d in DefenseType:
    print('  -', d.name)

## 5.3 Apply Individual Defenses

In [ ]:
defense_results = {}

for defense_type in DefenseType:
    defended = pipeline.defend_detections(attacked, defense_type, ground_truth)
    defense_results[defense_type.name] = defended
    
    benign_bearings = detections[3]['bearing'].dropna().values
    attacked_bearings = attacked[3]['bearing'].dropna().values
    defended_bearings = defended[3]['bearing'].dropna().values
    
    attack_shift = np.mean(np.abs(attacked_bearings[:len(benign_bearings)] - benign_bearings[:len(attacked_bearings)]))
    defense_shift = np.mean(np.abs(defended_bearings[:len(benign_bearings)] - benign_bearings[:len(defended_bearings)]))
    recovery = (1 - defense_shift / attack_shift) * 100 if attack_shift > 0 else 100
    
    print(defense_type.name + ': Attack shift=' + str(round(attack_shift, 4)) + ', Defense shift=' + str(round(defense_shift, 4)) + ', Recovery=' + str(round(recovery, 1)) + '%')

## 5.4 Visualize Defense Recovery

In [ ]:
best_defense = DefenseType.TEMPORAL_CONSISTENCY
defended = defense_results[best_defense.name]

fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(detections[3]['x_piren'], detections[3]['y_piren'], s=5, alpha=0.5, c='blue', label='Benign')
ax.scatter(attacked[3]['x_piren'], attacked[3]['y_piren'], s=5, alpha=0.5, c='red', label='Attacked')
ax.scatter(defended[3]['x_piren'], defended[3]['y_piren'], s=5, alpha=0.5, c='green', label='Defended')
ax.set_title('IR Camera Defense: ' + best_defense.name)
ax.set_xlabel('East (m)')
ax.set_ylabel('North (m)')
ax.legend()
ax.grid(True)
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 5.5 Compare All Defenses

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for idx, (defense_name, defended) in enumerate(defense_results.items()):
    ax = axes[idx]
    ax.scatter(detections[3]['x_piren'], detections[3]['y_piren'], s=3, alpha=0.3, c='blue', label='Benign')
    ax.scatter(attacked[3]['x_piren'], attacked[3]['y_piren'], s=3, alpha=0.3, c='red', label='Attacked')
    ax.scatter(defended[3]['x_piren'], defended[3]['y_piren'], s=3, alpha=0.3, c='green', label='Defended')
    ax.set_title(defense_name.replace('_', ' '))
    ax.set_xlabel('East (m)')
    ax.set_ylabel('North (m)')
    ax.legend()
    ax.grid(True)
    ax.set_aspect('equal')

plt.suptitle('Defense Comparison: IR Camera FGSM Attack', fontsize=14)
plt.tight_layout()
plt.show()

## 5.6 Certified Defense Analysis

In [ ]:
from defenses.defense_mechanisms import CertifiedDefense

radii = [0.01, 0.02, 0.05, 0.1, 0.15, 0.2]
certified_results = []

for radius in radii:
    cert_def = CertifiedDefense(certified_radius=radius)
    defended = cert_def.defend(attacked, ground_truth)
    benign_bearings = detections[3]['bearing'].dropna().values[:100]
    defended_bearings = defended[3]['bearing'].dropna().values[:100]
    rmse = np.sqrt(np.mean((defended_bearings - benign_bearings)**2))
    certified_results.append({'radius': radius, 'rmse': rmse})
    print('Radius=' + str(radius) + ': RMSE=' + str(round(rmse, 4)))

radii_vals = [r['radius'] for r in certified_results]
rmses = [r['rmse'] for r in certified_results]

plt.figure(figsize=(10, 6))
plt.plot(radii_vals, rmses, 'bo-', linewidth=2, markersize=10)
plt.xlabel('Certified Radius')
plt.ylabel('RMSE (rad)')
plt.title('Certified Defense: Radius vs Recovery Quality')
plt.grid(True)
plt.show()

## 5.7 Adversarial Training Defense

In [ ]:
from defenses.defense_mechanisms import AdversarialTraining

adv_train = AdversarialTraining(augmentation_ratio=0.3)
adv_train.train(detections, attacked)
defended_adv = adv_train.defend(attacked)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 6))

ax1.scatter(detections[3]['x_piren'], detections[3]['y_piren'], s=5, alpha=0.5, c='blue')
ax1.set_title('Benign')
ax1.set_xlabel('East (m)')
ax1.set_ylabel('North (m)')
ax1.grid(True)
ax1.set_aspect('equal')

ax2.scatter(attacked[3]['x_piren'], attacked[3]['y_piren'], s=5, alpha=0.5, c='red')
ax2.set_title('Attacked (FGSM)')
ax2.set_xlabel('East (m)')
ax2.set_ylabel('North (m)')
ax2.grid(True)
ax2.set_aspect('equal')

ax3.scatter(defended_adv[3]['x_piren'], defended_adv[3]['y_piren'], s=5, alpha=0.5, c='green')
ax3.set_title('Adversarial Training Defense')
ax3.set_xlabel('East (m)')
ax3.set_ylabel('North (m)')
ax3.grid(True)
ax3.set_aspect('equal')

plt.suptitle('Adversarial Training Defense', fontsize=14)
plt.tight_layout()
plt.show()